# JSON LabelMe to GeoJSON

In [2]:
import json
import rasterio
import os
import glob
from pathlib import Path

def labelme_to_geojson(input_json_path, input_tif_path, output_geojson_path):
    # Read LabelMe JSON file
    with open(input_json_path, 'r') as f:
        labelme_data = json.load(f)
    
    # Read georeference information from TIF
    with rasterio.open(input_tif_path) as src:
        # Get the transformation matrix
        transform = src.transform
        # Get the CRS
        crs = src.crs.to_string()
    
    # Initialize GeoJSON structure
    geojson = {
        "type": "FeatureCollection",
        "name": Path(input_json_path).stem,  # Use filename as name
        "crs": {
            "type": "name",
            "properties": {
                "name": f"urn:ogc:def:crs:EPSG::{src.crs.to_epsg()}"
            }
        },
        "features": []
    }

    # Convert each polygon annotation to GeoJSON feature
    for shape in labelme_data['shapes']:
        if shape['shape_type'] == 'polygon':

            confidence_score = shape["score"]

            # Convert pixel coordinates to geographic coordinates
            coords = []
            for point in shape['points']:
                # Convert from pixel coordinates to geographic coordinates using the transform
                geo_x, geo_y = transform * (point[0], point[1])
                coords.append([geo_x, geo_y])
            
            # Close the polygon by adding the first point at the end
            coords.append(coords[0])
            
            # Create GeoJSON feature
            feature = {
                "type": "Feature",
                "properties": {
                    "Confidence_score": confidence_score
                    #"label": shape['label']
                },
                "geometry": {
                    "type": "Polygon",
                    "coordinates": [coords]
                }
            }
            
            geojson['features'].append(feature)

    # Write GeoJSON file
    with open(output_geojson_path, 'w') as f:
        json.dump(geojson, f, indent=2)

def process_directory(input_json_dir, input_tif_dir, output_dir):
    """
    Memproses semua file JSON LabelMe dalam direktori, mencocokkannya dengan 
    file TIF yang sesuai dari direktori lain, dan mengonversinya ke GeoJSON.
    """
    # Buat direktori output jika belum ada
    os.makedirs(output_dir, exist_ok=True)
    
    # Dapatkan semua file JSON di direktori input
    json_files = glob.glob(os.path.join(input_json_dir, "*.json"))
    
    if not json_files:
        print(f"Warning: Tidak ada file .json yang ditemukan di {input_json_dir}")
        return

    print(f"Menemukan {len(json_files)} file JSON untuk diproses...")
    
    # Proses setiap file
    for json_file in json_files:
        # Dapatkan nama file dasar (misal: 'gambar_01')
        base_name = Path(json_file).stem
        
        # === PERBAIKAN DI SINI ===
        # Bangun path file TIF yang sesuai menggunakan input_tif_dir
        tif_file = os.path.join(input_tif_dir, base_name + '.tif')
        
        # Lewati jika file TIF yang cocok tidak ada
        if not os.path.exists(tif_file):
            print(f"Warning: Tidak ada TIF file yang cocok di {tif_file} untuk {Path(json_file).name}")
            continue
        
        # Bangun nama file output GeoJSON
        output_file = os.path.join(output_dir, base_name + '.geojson')
        
        try:
            print(f"Processing {Path(json_file).name}...")
            # Panggil fungsi konversi Anda
            labelme_to_geojson(json_file, tif_file, output_file)
            print(f"Successfully created {Path(output_file).name}")
        except Exception as e:
            print(f"Error processing {Path(json_file).name}: {str(e)}")

In [3]:
# Example usage
input_json_dir  = "./sample/tiles"
input_tif_dir   = "./sample/tiles"
output_dir      = "./sample/tiles/geojson"

process_directory(input_json_dir, input_tif_dir, output_dir)

Menemukan 167 file JSON untuk diproses...
Processing tile_0_0.json...
Successfully created tile_0_0.geojson
Processing tile_0_1.json...
Successfully created tile_0_1.geojson
Processing tile_0_10.json...
Successfully created tile_0_10.geojson
Processing tile_0_11.json...
Successfully created tile_0_11.geojson
Processing tile_0_12.json...
Successfully created tile_0_12.geojson
Processing tile_0_2.json...
Successfully created tile_0_2.geojson
Processing tile_0_3.json...
Successfully created tile_0_3.geojson
Processing tile_0_4.json...
Successfully created tile_0_4.geojson
Processing tile_0_5.json...
Successfully created tile_0_5.geojson
Processing tile_0_6.json...
Successfully created tile_0_6.geojson
Processing tile_0_7.json...
Successfully created tile_0_7.geojson
Processing tile_0_8.json...
Successfully created tile_0_8.geojson
Processing tile_0_9.json...
Successfully created tile_0_9.geojson
Processing tile_10_0.json...
Successfully created tile_10_0.geojson
Processing tile_10_1.json.

# NMS Standar (Ball et al.)

In [4]:
import os
import glob
import rasterio
import pandas as pd
import geopandas as gpd
from shapely.geometry import box

def calc_iou(shape1, shape2):
    """Menghitung persentase tumpang tindih (IoU) dari dua poligon."""
    intersection_area = shape1.intersection(shape2).area
    union_area = shape1.union(shape2).area
    return intersection_area / union_area if union_area > 0 else 0

def clean_crowns(crowns: gpd.GeoDataFrame,
                 iou_threshold: float = 0.5,
                 confidence_threshold: float = 0.5,
                 area_threshold: float = 2.0,
                 field: str = "Confidence_score") -> gpd.GeoDataFrame:
    """Membersihkan poligon tumpang tindih (NMS) dengan IoU tertentu."""
    
    # Filter yang tidak valid dan ukurannya terlalu kecil (artefak)
    crowns = crowns[~crowns.is_empty & crowns.is_valid]
    crowns = crowns[crowns.area > area_threshold]
    crowns.reset_index(drop=True, inplace=True)

    cleaned_crowns = []
    print(f"Memulai pembersihan {len(crowns)} poligon...")

    for index, row in crowns.iterrows():
        if index % 1000 == 0 and index > 0:
            print(f"  Memproses {index} / {len(crowns)} poligon...")

        intersecting_rows = crowns[crowns.intersects(row.geometry)]

        if len(intersecting_rows) > 1:
            iou_values = intersecting_rows.geometry.apply(lambda x: calc_iou(row.geometry, x))
            intersecting_rows = intersecting_rows.assign(iou=iou_values)

            # Ambil 1 poligon dengan score tertinggi di antara yang overlap
            match = intersecting_rows[intersecting_rows["iou"] > iou_threshold].nlargest(1, field)

            if len(match) > 0 and match["iou"].iloc[0] < 1:
                continue
        else:
            match = row.to_frame().T

        cleaned_crowns.append(match)

    if cleaned_crowns:
        crowns_out = pd.concat(cleaned_crowns, ignore_index=True)
        if "iou" in crowns_out.columns:
            crowns_out = crowns_out.drop("iou", axis=1)
    else:
        crowns_out = gpd.GeoDataFrame()

    if not isinstance(crowns_out, gpd.GeoDataFrame):
        crowns_out = gpd.GeoDataFrame(crowns_out, crs=crowns.crs)
    else:
        crowns_out = crowns_out.set_crs(crowns.crs, allow_override=True)

    # Filter skor akhir
    if confidence_threshold > 0 and field in crowns_out.columns:
        crowns_out[field] = pd.to_numeric(crowns_out[field])
        crowns_out = crowns_out[crowns_out[field] >= confidence_threshold]

    return crowns_out.reset_index(drop=True)

def filter_edge_crowns(gdf_tile, tif_path, shift=1.0):
    """
    MEMBUAT MEKANISME SHIFT 1 METER MILIK BALL:
    Membaca batas tepi (bounding box) dari file .tif asli, mengecilkannya 
    sejauh `shift` meter, dan membuang semua pohon yang menyentuh/melewati batas tersebut.
    """
    with rasterio.open(tif_path) as src:
        # Ambil batas luar gambar .tif dalam satuan meter (CRS geospasial)
        minx, miny, maxx, maxy = src.bounds.left, src.bounds.bottom, src.bounds.right, src.bounds.top
        
    # Buat kotak (bounding box) dan kecilkan sebanyak 'shift' (default: 1 meter)
    shrunken_bbox = box(minx + shift, miny + shift, maxx - shift, maxy - shift)
    
    # Hanya simpan poligon pohon yang SEPENUHNYA BERADA DI DALAM kotak yang sudah dikecilkan
    gdf_filtered = gdf_tile[gdf_tile.geometry.within(shrunken_bbox)]
    
    return gdf_filtered

def stitch_and_clean_pipeline(geojson_dir: str, tif_dir: str, output_gpkg: str, shift: float = 1.0):
    geojson_files = glob.glob(os.path.join(geojson_dir, "*.geojson"))
    
    if not geojson_files:
        print(f"[!] Tidak ada file GeoJSON di {geojson_dir}")
        return

    print(f"Tahap 1: Stitching & Edge Bounding Box Shift ({shift}m) pada {len(geojson_files)} file...")
    
    gdfs = []
    for geojson_file in geojson_files:
        gdf = gpd.read_file(geojson_file)
        if gdf.empty:
            continue
            
        # Cari file .tif pasangannya (misal: tile_0_0.geojson -> tile_0_0.tif)
        base_name = os.path.splitext(os.path.basename(geojson_file))[0]
        tif_path = os.path.join(tif_dir, f"{base_name}.tif")
        
        if not os.path.exists(tif_path):
            print(f"  [!] Peringatan: File {base_name}.tif tidak ditemukan. Shift dilewati untuk file ini.")
            gdfs.append(gdf)
            continue
            
        # Terapkan mekanisme potong tepi 1 meter
        gdf_shifted = filter_edge_crowns(gdf, tif_path, shift=shift)
        gdfs.append(gdf_shifted)
        
    # Gabungkan (Stitch) semua tile yang sudah dipotong tepinya
    merged_gdf = pd.concat(gdfs, ignore_index=True)
    merged_gdf = gpd.GeoDataFrame(merged_gdf, crs=gdfs[0].crs)
    print(f"Total poligon setelah Stitching + Shift tepi: {len(merged_gdf)}")
    
    # Tahap 2: Proses pembersihan (NMS & Score Thresholding)
    print("\nTahap 2: Cleaning (NMS) & Filtering...")
    
    # Menjalankan pembersihan persis seperti urutan instruksi Anda
    cleaned_gdf = clean_crowns(
        crowns=merged_gdf, 
        iou_threshold=0.5,         # Sesuai langkah Anda
        confidence_threshold=0.5,  # Sesuai langkah Anda
        field="Confidence_score"   # Sesuaikan field score dari geojson Anda
    )
    
    print(f"Total poligon akhir (Siap ekspor): {len(cleaned_gdf)}")
    
    # Tahap 3: Simpan ke GPKG
    cleaned_gdf.to_file(output_gpkg, driver="GPKG")
    print(f"\n[v] Selesai! File bersih tersimpan di: {output_gpkg}")

In [5]:
# =======================================================
# CARA PENGGUNAAN
# =======================================================
SITE_PATH = "./sample/tiles/"

DIR_PREDICTIONS = os.path.join(SITE_PATH, "geojson") 
DIR_TILES_TIF   = os.path.join(SITE_PATH, "") # Folder tempat tile_0_0.tif berada
FILE_OUTPUT     = os.path.join(SITE_PATH, "tree_crowns_ball.gpkg")

# Jalankan pipeline penuh:
stitch_and_clean_pipeline(
    geojson_dir = DIR_PREDICTIONS, 
    tif_dir     = DIR_TILES_TIF, 
    output_gpkg = FILE_OUTPUT,
    shift       = 1.0  # Pemotongan tepi (margin pinggir gambar) sejauh 1 meter
)

Tahap 1: Stitching & Edge Bounding Box Shift (1.0m) pada 167 file...
Total poligon setelah Stitching + Shift tepi: 1805

Tahap 2: Cleaning (NMS) & Filtering...
Memulai pembersihan 1719 poligon...
  Memproses 1000 / 1719 poligon...
Total poligon akhir (Siap ekspor): 1391

[v] Selesai! File bersih tersimpan di: ./sample/tiles/tree_crowns_ball.gpkg
